# Libraries

In [2]:
import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo
from sklearn.datasets import fetch_california_housing

from tensorflow.keras import layers, models
import tensorflow as tf

from sklearn.metrics import confusion_matrix, f1_score, classification_report, accuracy_score, balanced_accuracy_score, ConfusionMatrixDisplay, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# own functions
from active_learning import random_sampling, uncertainty_sampling, label_data, active_learning_bc, active_learning_mc, sasla_bc, sasla_mc
from active_learning import balanced_weights, tune_params, retrieve_params, cv_classification, create_model

from time import time

# Models

## Breast Cancer

In [3]:
breast_cancer = fetch_ucirepo(id=17)

bcwd_df = pd.concat(
    [breast_cancer.data.features, breast_cancer.data.targets],
    axis=1
)
bcwd_df.head()

,radius1,texture1,perimeter1,area1,smoothness1,compactness1,concavity1,concave_points1,symmetry1,fractal_dimension1,...,texture3,perimeter3,area3,smoothness3,compactness3,concavity3,concave_points3,symmetry3,fractal_dimension3,Diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,M
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,M
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,M
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,M
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,M


In [4]:
X = breast_cancer.data.features.to_numpy()
y = (breast_cancer.data.targets).iloc[:, 0].map({"B": 0, "M": 1}).astype("int32").to_numpy()

X_pool, X_test, y_pool, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=51
)

In [4]:
acts = ['relu', 'leaky_relu', 'elu', 'gelu', 'swish']
balance = [False, True]
name = "base"
for state in balance:
    if state:
        name = 'balanced'
    else:
        name = 'base'

    for activation in acts:
        base_params = tune_params(X_pool, y_pool,
                                  loss='binary_crossentropy',
                                  out_units=1, out_act='sigmoid',
                                  k=5, activation=activation,
                                  balanced=False,
                                  random_state=42,
                                  n_trials=50,
                                  export=True, file = f'params/breast_cancer/{name}_{activation}.json'
        )

[I 2026-09-25 19:56:55,369] A new study created in memory with name: no-name-455c1104-dff1-40ab-b8ea-5ad80ccf7a06


[W 2026-09-25 19:57:01,649] Trial 0 failed with parameters: {'hidden_layer': 17, 'learning_rate': 0.0025303472686758082, 'momentum': 0.05187663174830328, 'weight_decay': 0.0006069634458336872} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "C:\Users\dawid\AppData\Roaming\Python\Python311\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\dawid\OneDrive - Stellenbosch University\Documents\Universiteit\4th year\ML\assignment3\ML441_Assigment3\active_learning.py", line 582, in obj
    model.fit(
  File "C:\Users\dawid\AppData\Roaming\Python\Python311\site-packages\keras\src\utils\traceback_utils.py", line 117, in error_handler
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\dawid\AppData\Roaming\Python\Python311\site-packages\keras\src\backend\tensorflow\trainer.py", line 375, in fit
    for begin_step, end_step,

KeyboardInterrupt: 

In [4]:
acts = ['relu', 'leaky_relu', 'elu', 'gelu', 'swish']
balance = [False, True]

results = []
name = "base"
for state in balance:
    if state:
        name = 'balanced'
    else:
        name = 'base'

    for activation in acts:
        setup = {'activation': activation, 'balanced': state}
        alpha = 0
        params = retrieve_params(f'params/breast_cancer/{name}_{activation}.json')
        if activation == 'leaky_relu':
            alpha = params["leaky_relu_alpha"]
        elif activation == 'elu':
            alpha = params["elu_alpha"]
        result = cv_classification(X_pool, y_pool, out_units=1, out_act='sigmoid', loss="binary_crossentropy", k=5, random_state=42, 
                  hidden_units=params['hidden_layer'], activation=activation, alpha=alpha, 
                  learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], balanced=state)
        for key, val in result.items():
            setup[key] = val
    
        results.append(setup)

results_df = pd.DataFrame(results)
results_df.round(4).to_csv(f"results/breast_cancer/cv_validation.csv", index=False)

In [13]:
results_df = pd.read_csv("results/breast_cancer/cv_validation.csv")
results_df[results_df['balanced'] == False]

,activation,balanced,auc,acc,f1
0,relu,False,0.9926,0.9648,0.9624
1,leaky_relu,False,0.9909,0.9670,0.9647
2,elu,False,0.9922,0.9736,0.9715
3,gelu,False,0.9921,0.9692,0.9671
4,swish,False,0.9922,0.9714,0.9693


In [16]:
activation = 'elu'
params = retrieve_params(f'params/breast_cancer/base_{activation}.json')
scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)

model = create_model(X_train.shape[1], hidden_units=params['hidden_layer'], activation=activation, alpha=params['elu_alpha'], out_units=1, out_act='sigmoid', 
                 learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], loss='binary_crossentropy')
# params['learning_rate']
history = []
runs = 100
epochs = 200 // runs
fit_kwargs = {
    "epochs": epochs,
    "batch_size": 32,
    # "validation_split": 0.15,
    # "callbacks": [early_stopping],
    "verbose": 0
}
tic = time()
for i in range(runs):
    iteration = {'cumulative epochs': epochs * (i + 1)}
    iter_hist = model.fit(
        X_train,
        y_pool,
        **fit_kwargs
    )

    train_loss = iter_hist.history['loss']
    weights = model.get_weights()

    if not all(np.all(np.isfinite(w)) for w in weights):
        print(f"Model became unstable at iteration {i + 1}")
        break

    y_prob = model.predict(X_val, verbose=0).ravel()

    if not np.all(np.isfinite(y_prob)):
        print(f"NaN/Inf predictions at iteration {i + 1}")
        break
        
    y_pred = (y_prob >= 0.5).astype(int)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    auc = roc_auc_score(y_test, y_prob)
    toc = time()
    
    iteration['Training loss'] = train_loss[-1]
    iteration['Accuracy'] = acc
    iteration['macro F1'] = f1
    iteration['ROC-AUC'] = auc
    iteration['time'] = toc - tic
    history.append(iteration)
    print(f"Iteration {i+1}: ")
    for l in range(len(train_loss)):
        print(f"\t{train_loss[l]}")

run_df = pd.DataFrame(history)
run_df.round(4).to_csv(f"results/breast_cancer/passive_learning.csv", index=False)

Iteration 1: 
	0.31497400999069214
	0.12422307580709457
Iteration 2: 
	0.10408703982830048
	0.09339655190706253
Iteration 3: 
	0.08665139973163605
	0.0814039409160614
Iteration 4: 
	0.07830493152141571
	0.0757574513554573
Iteration 5: 
	0.07305547595024109
	0.07225505262613297
Iteration 6: 
	0.07031994313001633
	0.06819263845682144
Iteration 7: 
	0.06699642539024353
	0.0656541958451271
Iteration 8: 
	0.06443711370229721
	0.06362545490264893
Iteration 9: 
	0.06300844252109528
	0.06177350878715515
Iteration 10: 
	0.06105828657746315
	0.05996248871088028
Iteration 11: 
	0.05970074608922005
	0.05898311734199524
Iteration 12: 
	0.0583430752158165
	0.057868074625730515
Iteration 13: 
	0.05711057409644127
	0.05644910782575607
Iteration 14: 
	0.05618635565042496
	0.0554165318608284
Iteration 15: 
	0.05479390174150467
	0.05460323393344879
Iteration 16: 
	0.05417875200510025
	0.05368169769644737
Iteration 17: 
	0.05365431308746338
	0.053356196731328964
Iteration 18: 
	0.05275879055261612
	0.0521

In [7]:
activation = 'elu'
params = retrieve_params(f'params/breast_cancer/base_{activation}.json')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)

history = active_learning_bc(X_train, y_pool, X_val, y_test, random_sampling, num_iter=100, unlab_size=0.9, random_state=100, verbose=True,
                               hidden_units=params['hidden_layer'], activation=activation, alpha=params['elu_alpha'], balanced=False,
                               learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], loss='binary_crossentropy' 
)

run_df = pd.DataFrame(history)
run_df.round(4).to_csv(f"results/breast_cancer/uncertainty_sampling.csv", index=False)

Iteration 1: 45 labelled instances
	Training loss = 0.40008431673049927

Iteration 2: 49 labelled instances
	Training loss = 0.22699640691280365

Iteration 3: 53 labelled instances
	Training loss = 0.15667562186717987

Iteration 4: 57 labelled instances
	Training loss = 0.11602528393268585

Iteration 5: 61 labelled instances
	Training loss = 0.10972920060157776

Iteration 6: 65 labelled instances
	Training loss = 0.10710886120796204

Iteration 7: 69 labelled instances
	Training loss = 0.09189688414335251

Iteration 8: 73 labelled instances
	Training loss = 0.07976818829774857

Iteration 9: 77 labelled instances
	Training loss = 0.07067061960697174

Iteration 10: 81 labelled instances
	Training loss = 0.06417559087276459

Iteration 11: 85 labelled instances
	Training loss = 0.05795629322528839

Iteration 12: 89 labelled instances
	Training loss = 0.05915168300271034

Iteration 13: 93 labelled instances
	Training loss = 0.054313674569129944

Iteration 14: 97 labelled instances
	Training 

In [8]:
activation = 'elu'
params = retrieve_params(f'params/breast_cancer/base_{activation}.json')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)
for beta in [0.9]:
    history = sasla_bc(X_train, y_pool, X_val, y_test, num_iter=100, unlab_size=0.9, random_state=100, verbose=True,
                                   hidden_units=params['hidden_layer'], activation=activation, alpha=params['elu_alpha'], balanced=False,
                                   learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], beta=beta)
    
    run_df = pd.DataFrame(history)
    run_df.round(4).to_csv(f"results/breast_cancer/sasla_{beta}.csv", index=False)

Iteration 1: 455 labelled instances
	Accuracy = 0.9824561403508771
	macro F1 = 0.9811507936507937
	AUC = 0.9990079365079365

Iteration 2: 350 labelled instances
	Accuracy = 0.9824561403508771
	macro F1 = 0.9811507936507937
	AUC = 0.9993386243386243

Iteration 3: 298 labelled instances
	Accuracy = 0.9824561403508771
	macro F1 = 0.9811507936507937
	AUC = 0.9993386243386243

Iteration 4: 263 labelled instances
	Accuracy = 0.9824561403508771
	macro F1 = 0.9811507936507937
	AUC = 0.9993386243386243

Iteration 5: 237 labelled instances
	Accuracy = 0.9824561403508771
	macro F1 = 0.9811507936507937
	AUC = 0.9993386243386243

Iteration 6: 223 labelled instances
	Accuracy = 0.9912280701754386
	macro F1 = 0.9905276277523889
	AUC = 0.9996693121693122

Iteration 7: 216 labelled instances
	Accuracy = 0.9912280701754386
	macro F1 = 0.9905276277523889
	AUC = 0.9996693121693122

Iteration 8: 204 labelled instances
	Accuracy = 0.9912280701754386
	macro F1 = 0.9905276277523889
	AUC = 0.9996693121693122



## Dry beans

In [9]:
dry_bean = fetch_ucirepo(id=602)
bean_df = pd.concat(
    [dry_bean.data.features, dry_bean.data.targets],
    axis=1
)
bean_df.head()

,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRatio,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,Roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272751,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER


In [10]:
X = dry_bean.data.features.to_numpy()
encoder = LabelEncoder()
y = dry_bean.data.targets.to_numpy().ravel()

n_classes = len(np.unique(y))

y = encoder.fit_transform(y)
bean_map = encoder.classes_
n_classes = len(np.unique(y))

X_pool, X_test, y_pool, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=51
)

In [7]:
acts = ['relu', 'leaky_relu', 'elu', 'gelu', 'swish']
name = "base"
for activation in acts:
    base_params = tune_params(X_pool, y_pool,
                              loss='sparse_categorical_crossentropy',
                              out_units=n_classes, out_act='softmax',
                              k=5, activation=activation,
                              balanced=False,
                              random_state=42,
                              n_trials=80,
                              export=True, file = f'params/dry_bean/{name}_{activation}.json'
    )

[I 2026-09-24 08:58:49,653] A new study created in memory with name: no-name-f7990ee7-4d94-46ba-ad02-462e9c640964
[I 2026-09-24 09:03:42,963] Trial 0 finished with value: 0.2359083592891693 and parameters: {'hidden_layer': 31, 'learning_rate': 0.0010504234588356461, 'momentum': 0.5672917461412101, 'weight_decay': 0.00025515089599309465}. Best is trial 0 with value: 0.2359083592891693.
[I 2026-09-24 09:08:12,182] Trial 1 finished with value: 0.20321372151374817 and parameters: {'hidden_layer': 74, 'learning_rate': 0.006456565189986694, 'momentum': 0.252843329131544, 'weight_decay': 1.3376349549684816e-05}. Best is trial 1 with value: 0.20321372151374817.
[I 2026-09-24 09:12:47,467] Trial 2 finished with value: 0.24281105399131775 and parameters: {'hidden_layer': 26, 'learning_rate': 0.0010622162477497206, 'momentum': 0.514092887324837, 'weight_decay': 4.989903460203718e-05}. Best is trial 1 with value: 0.20321372151374817.
[I 2026-09-24 09:14:24,182] Trial 3 finished with value: 0.20411

{'hidden_layer': 107, 'learning_rate': 0.007758378811390079, 'momentum': 0.9414639060544645, 'weight_decay': 0.0001097303171328828}
0.19380834698677063


[I 2026-09-24 10:34:40,683] Trial 0 finished with value: 0.603901743888855 and parameters: {'hidden_layer': 38, 'learning_rate': 0.0002673158232409953, 'momentum': 0.032274798726985035, 'weight_decay': 4.936605963515345e-05, 'leaky_relu_alpha': 0.003109727808447892}. Best is trial 0 with value: 0.603901743888855.
[I 2026-09-24 10:39:23,412] Trial 1 finished with value: 0.23986172676086426 and parameters: {'hidden_layer': 83, 'learning_rate': 0.0012903958116131534, 'momentum': 0.3565891977479561, 'weight_decay': 5.9950085693023085e-05, 'leaky_relu_alpha': 0.008350776258629306}. Best is trial 1 with value: 0.23986172676086426.
[I 2026-09-24 10:44:11,391] Trial 2 finished with value: 0.41579437255859375 and parameters: {'hidden_layer': 78, 'learning_rate': 0.0003941285306908751, 'momentum': 0.20072690389699424, 'weight_decay': 0.001403334804497561, 'leaky_relu_alpha': 0.004809521334064588}. Best is trial 1 with value: 0.23986172676086426.
[I 2026-09-24 10:49:39,590] Trial 3 finished with 

{'hidden_layer': 126, 'learning_rate': 0.038499144390277434, 'momentum': 0.8888210220929265, 'weight_decay': 5.114730565238182e-06, 'leaky_relu_alpha': 0.020452474706357127}
0.19284117221832275


[I 2026-09-24 12:28:10,555] Trial 0 finished with value: 0.20084130764007568 and parameters: {'hidden_layer': 58, 'learning_rate': 0.0094677616697529, 'momentum': 0.9291702662166553, 'weight_decay': 6.679133460540766e-05, 'elu_alpha': 1.2326761011141363}. Best is trial 0 with value: 0.20084130764007568.
[I 2026-09-24 12:29:37,334] Trial 1 finished with value: 0.19823968410491943 and parameters: {'hidden_layer': 70, 'learning_rate': 0.06869919389678714, 'momentum': 0.7515212120359361, 'weight_decay': 6.7135691451640234e-06, 'elu_alpha': 0.10652297231932507}. Best is trial 1 with value: 0.19823968410491943.
[I 2026-09-24 12:33:20,755] Trial 2 finished with value: 0.20361976325511932 and parameters: {'hidden_layer': 40, 'learning_rate': 0.005478168211603529, 'momentum': 0.7312758087345064, 'weight_decay': 4.161424936052871e-06, 'elu_alpha': 1.1496989473310768}. Best is trial 1 with value: 0.19823968410491943.
[I 2026-09-24 12:39:18,291] Trial 3 finished with value: 0.24139642715454102 and

...

{'hidden_layer': 86, 'learning_rate': 0.06916135455563951, 'momentum': 0.8404367717780885, 'weight_decay': 0.0005393788572166543}
0.19460535049438477


[I 2026-09-24 15:43:04,131] Trial 0 finished with value: 0.21250960230827332 and parameters: {'hidden_layer': 46, 'learning_rate': 0.0061649117765305295, 'momentum': 0.026327865999117597, 'weight_decay': 3.4252575969670253e-06}. Best is trial 0 with value: 0.21250960230827332.
[I 2026-09-24 15:49:11,801] Trial 1 finished with value: 0.5587712526321411 and parameters: {'hidden_layer': 94, 'learning_rate': 0.00014306667156999147, 'momentum': 0.5856017619536737, 'weight_decay': 6.598749807683256e-05}. Best is trial 0 with value: 0.21250960230827332.
[I 2026-09-24 15:51:33,281] Trial 2 finished with value: 0.19777460396289825 and parameters: {'hidden_layer': 86, 'learning_rate': 0.01000314757145471, 'momentum': 0.915372569690932, 'weight_decay': 6.87168975312532e-05}. Best is trial 2 with value: 0.19777460396289825.
[I 2026-09-24 15:56:42,992] Trial 3 finished with value: 0.22301384806632996 and parameters: {'hidden_layer': 33, 'learning_rate': 0.006216506625991501, 'momentum': 0.498443313

{'hidden_layer': 57, 'learning_rate': 0.026731908629622256, 'momentum': 0.7455897737662243, 'weight_decay': 3.99937172643707e-06}
0.1956712007522583


[I 2026-09-24 17:48:15,933] Trial 0 finished with value: 0.374485582113266 and parameters: {'hidden_layer': 70, 'learning_rate': 0.00024927065531605896, 'momentum': 0.6492330335921463, 'weight_decay': 1.2087112005111532e-06}. Best is trial 0 with value: 0.374485582113266.
[I 2026-09-24 17:53:57,007] Trial 1 finished with value: 0.3847348690032959 and parameters: {'hidden_layer': 93, 'learning_rate': 0.000497296738567034, 'momentum': 0.24224269611933166, 'weight_decay': 0.00013289185475650556}. Best is trial 0 with value: 0.374485582113266.
[I 2026-09-24 17:56:44,662] Trial 2 finished with value: 0.21049609780311584 and parameters: {'hidden_layer': 17, 'learning_rate': 0.013905546486568707, 'momentum': 0.5172550951043268, 'weight_decay': 2.8984542946793415e-06}. Best is trial 2 with value: 0.21049609780311584.
[I 2026-09-24 18:00:36,074] Trial 3 finished with value: 0.2036658525466919 and parameters: {'hidden_layer': 111, 'learning_rate': 0.003340192661172581, 'momentum': 0.823304634779

KeyboardInterrupt: 

In [5]:
acts = ['relu', 'leaky_relu', 'elu']

results = []
name = "base"

for activation in acts:
    setup = {'activation': activation, 'balanced': False}
    alpha = 0
    params = retrieve_params(f'params/dry_bean/{name}_{activation}.json')
    if activation == 'leaky_relu':
        alpha = params["leaky_relu_alpha"]
    elif activation == 'elu':
        alpha = params["elu_alpha"]
    result = cv_classification(X_pool, y_pool, 
               out_units=n_classes, out_act='softmax', loss='sparse_categorical_crossentropy', k=5, random_state=42, 
               hidden_units=params['hidden_layer'], activation=activation, alpha=alpha, 
               learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], balanced=False)
    for key, val in result.items():
        setup[key] = val

    results.append(setup)

results_df = pd.DataFrame(results)
results_df.round(4).to_csv(f"results/dry_bean/cv_validation.csv", index=False)

In [9]:
results_df = pd.read_csv("results/dry_bean/cv_validation.csv")
results_df

,activation,balanced,auc,acc,f1
0,relu,False,0.9955,0.9288,0.9398
1,leaky_relu,False,0.9954,0.9277,0.9391
2,elu,False,0.9955,0.9284,0.9393


In [ ]:
activation = 'relu'
params = retrieve_params(f'params/dry_bean/base_{activation}.json')
scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)

model = create_model(X_train.shape[1], hidden_units=params['hidden_layer'], activation=activation, alpha=0, out_units=n_classes, out_act='softmax', 
                 learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], loss='sparse_categorical_crossentropy')
# params['learning_rate']
history = []
runs = 100
epochs = 200 // runs
fit_kwargs = {
    "epochs": epochs,
    "batch_size": 32,
    # "validation_split": 0.15,
    # "callbacks": [early_stopping],
    "verbose": 0
}
tic = time()
for i in range(runs):
    iteration = {'cumulative epochs': epochs * (i + 1)}
    iter_hist = model.fit(
        X_train,
        y_pool,
        **fit_kwargs
    )

    train_loss = iter_hist.history['loss']
    weights = model.get_weights()

    if not all(np.all(np.isfinite(w)) for w in weights):
        print(f"Model became unstable at iteration {i + 1}")
        break

    y_prob = model.predict(X_val, verbose=0)

    if not np.all(np.isfinite(y_prob)):
        print(f"NaN/Inf predictions at iteration {i + 1}")
        break
        
    y_pred = np.argmax(y_prob, axis=1)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    auc = roc_auc_score(y_test, y_prob, average='macro', multi_class='ovr')
    toc = time()
    
    iteration['Training loss'] = train_loss[-1]
    iteration['Accuracy'] = acc
    iteration['macro F1'] = f1
    iteration['ROC-AUC'] = auc
    iteration['time'] = toc - tic
    history.append(iteration)
    print(f"Iteration {i+1}: ")
    for l in range(len(train_loss)):
        print(f"\t{train_loss[l]}")

run_df = pd.DataFrame(history)
run_df.round(4).to_csv(f"results/dry_bean/passive_learning.csv", index=False)

In [11]:
activation = 'relu'
params = retrieve_params(f'params/dry_bean/base_{activation}.json')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)

history = active_learning_mc(X_train, y_pool, X_val, y_test, random_sampling, num_iter=100, unlab_size=0.9, random_state=100, verbose=True,
                               hidden_units=params['hidden_layer'], activation=activation, alpha=0, out_units=n_classes, balanced=False,
                               learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'] 
)

run_df = pd.DataFrame(history)
run_df.round(4).to_csv(f"results/dry_bean/uncertainty_sampling.csv", index=False)

Iteration 1: 1088 labelled instances
	Accuracy = 0.8813808299669482
	macro F1 = 0.8911188677631052
	AUC = 0.990251425608186

Iteration 2: 1186 labelled instances
	Accuracy = 0.9001101726037458
	macro F1 = 0.9107906707820641
	AUC = 0.9932912494900753

Iteration 3: 1284 labelled instances
	Accuracy = 0.9092912229159016
	macro F1 = 0.9208488707035366
	AUC = 0.9938116765912687

Iteration 4: 1382 labelled instances
	Accuracy = 0.9122291590157914
	macro F1 = 0.9227995489553352
	AUC = 0.9942728911874805

Iteration 5: 1480 labelled instances
	Accuracy = 0.9177377892030848
	macro F1 = 0.9291942837619205
	AUC = 0.9944312963420298

Iteration 6: 1578 labelled instances
	Accuracy = 0.9103929489533603
	macro F1 = 0.9226187201645379
	AUC = 0.994367110685262

Iteration 7: 1676 labelled instances
	Accuracy = 0.9125964010282777
	macro F1 = 0.9240986649605885
	AUC = 0.994608292751615

Iteration 8: 1774 labelled instances
	Accuracy = 0.9170033051781123
	macro F1 = 0.9278160942797624
	AUC = 0.9947930819512

In [12]:
activation = 'relu'
params = retrieve_params(f'params/dry_bean/base_{activation}.json')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)
for beta in [0.9]:
    history = sasla_mc(X_train, y_pool, X_val, y_test, num_iter=100, unlab_size=0.9, random_state=100, verbose=True,
                                   hidden_units=params['hidden_layer'], activation=activation, alpha=0, out_units=n_classes, balanced=False,
                                   learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], beta=beta)
    
    run_df = pd.DataFrame(history)
    run_df.round(4).to_csv(f"results/dry_bean/sasla_{beta}.csv", index=False)

Iteration 1: 10888 labelled instances
	Accuracy = 0.9236136614028645
	macro F1 = 0.9352480886269943
	AUC = 0.9952682430632374

Iteration 2: 6769 labelled instances
	Accuracy = 0.922879177377892
	macro F1 = 0.9344069204069907
	AUC = 0.9952337415320337

Iteration 3: 6317 labelled instances
	Accuracy = 0.9206757253029747
	macro F1 = 0.9329518189919968
	AUC = 0.9952427420973337

Iteration 4: 6363 labelled instances
	Accuracy = 0.9269188395152406
	macro F1 = 0.9372744868545337
	AUC = 0.9954562717913088

Iteration 5: 6167 labelled instances
	Accuracy = 0.9258171134777818
	macro F1 = 0.9361983700184542
	AUC = 0.9956172788518913

Iteration 6: 6151 labelled instances
	Accuracy = 0.9258171134777818
	macro F1 = 0.9362081979137613
	AUC = 0.9955711420490718

Iteration 7: 6020 labelled instances
	Accuracy = 0.9269188395152406
	macro F1 = 0.9377109043511966
	AUC = 0.9955865309014433

Iteration 8: 5906 labelled instances
	Accuracy = 0.9269188395152406
	macro F1 = 0.9386452104808033
	AUC = 0.9956148260

## Student dropout

In [13]:
student_dropout = fetch_ucirepo(id=697)
dropout_df = pd.concat(
    [student_dropout.data.features, student_dropout.data.targets],
    axis=1
)

# nominal features
nominal = ['Marital Status', 'Course', 'Previous qualification']
# Already international feature
# already parents' qual level and high cardinality
drop = ['Nacionality', "Mother's occupation", "Father's occupation"]

dropout_df.drop(columns=drop, inplace=True)
dropout_df = pd.get_dummies(
    dropout_df,
    columns=nominal,
    dtype=int
)

dropout_df.head()

,Application mode,Application order,Daytime/evening attendance,Previous qualification (grade),Mother's qualification,Father's qualification,Admission grade,Displaced,Educational special needs,Debtor,...,Previous qualification_10,Previous qualification_12,Previous qualification_14,Previous qualification_15,Previous qualification_19,Previous qualification_38,Previous qualification_39,Previous qualification_40,Previous qualification_42,Previous qualification_43
0,17,5,1,122.0,19,12,127.3,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,15,1,1,160.0,1,3,142.5,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,5,1,122.0,37,37,124.8,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,17,2,1,122.0,38,37,119.6,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,39,1,0,100.0,37,38,141.5,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
X = dropout_df.drop(columns=['Target']).to_numpy()
student_encoder = LabelEncoder()
y = student_dropout.data.targets.to_numpy().ravel()

y = student_encoder.fit_transform(y)
dropout_map = student_encoder.classes_
n_classes = len(np.unique(y))

X_pool, X_test, y_pool, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=51
)

In [6]:
# 'relu', 'leaky_relu', 
acts = ['elu']
name = 'base'
for activation in acts:
    base_params = tune_params(X_pool, y_pool,
                              loss='sparse_categorical_crossentropy',
                              out_units=n_classes, out_act='softmax',
                              k=5, activation=activation,
                              balanced=False,
                              random_state=42,
                              n_trials=80,
                              export=True, file = f'params/student_dropout/{name}_{activation}.json'
    )

[I 2026-09-26 00:31:29,054] A new study created in memory with name: no-name-d306407a-e149-4ee0-a73f-f1d8329ca88f


[I 2026-09-26 00:33:57,909] Trial 0 finished with value: 0.6028547883033752 and parameters: {'hidden_layer': 95, 'learning_rate': 0.0020656029332625546, 'momentum': 0.15247620238338155, 'weight_decay': 7.891012996392573e-06, 'elu_alpha': 0.4741044575563499}. Best is trial 0 with value: 0.6028547883033752.
[I 2026-09-26 00:35:40,558] Trial 1 finished with value: 0.6032333374023438 and parameters: {'hidden_layer': 75, 'learning_rate': 0.002147140331196139, 'momentum': 0.7682184833099834, 'weight_decay': 0.0030248390433163263, 'elu_alpha': 0.15745910494041737}. Best is trial 0 with value: 0.6028547883033752.
[I 2026-09-26 00:37:25,875] Trial 2 finished with value: 0.6042537689208984 and parameters: {'hidden_layer': 56, 'learning_rate': 0.002361436847544542, 'momentum': 0.6309182895177561, 'weight_decay': 0.0001362522257912527, 'elu_alpha': 0.4499536210785289}. Best is trial 0 with value: 0.6028547883033752.
[I 2026-09-26 00:39:17,150] Trial 3 finished with value: 0.5902585983276367 and pa

{'hidden_layer': 64, 'learning_rate': 0.009192486108599622, 'momentum': 0.042054381313093, 'weight_decay': 0.009767366643098496, 'elu_alpha': 0.621374494366071}
0.5736604928970337


In [11]:
acts = ['relu', 'leaky_relu', 'elu']
name = "base"
results = []
for activation in acts:
    setup = {'activation': activation, 'balanced': False}
    alpha = 0
    params = retrieve_params(f'params/student_dropout/{name}_{activation}.json')
    if activation == 'leaky_relu':
        alpha = params["leaky_relu_alpha"]
    elif activation == 'elu':
        alpha = params["elu_alpha"]
    result = cv_classification(X_pool, y_pool, out_units=n_classes, out_act='softmax', 
                    loss='sparse_categorical_crossentropy', k=5, random_state=42, 
                    hidden_units=params['hidden_layer'], activation=activation, alpha=alpha, 
                    learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], balanced=False)
    for key, val in result.items():
        setup[key] = val

    results.append(setup)

results_df = pd.DataFrame(results)
results_df.round(4).to_csv(f"results/student_dropout/cv_validation.csv", index=False)

In [14]:
results_df = pd.read_csv("results/student_dropout/cv_validation.csv")
results_df

,activation,balanced,auc,acc,f1
0,relu,False,0.8775,0.7691,0.6963
1,leaky_relu,False,0.8798,0.7708,0.6991
2,elu,False,0.8837,0.7697,0.6923


In [12]:
activation = 'leaky_relu'
params = retrieve_params(f'params/student_dropout/base_{activation}.json')
scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)

model = create_model(X_train.shape[1], hidden_units=params['hidden_layer'], activation=activation, alpha=params['leaky_relu_alpha'], out_units=n_classes, out_act='softmax', 
                 learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], loss='sparse_categorical_crossentropy')
# params['learning_rate']
history = []
runs = 100
epochs = 200 // runs
fit_kwargs = {
    "epochs": epochs,
    "batch_size": 32,
    # "validation_split": 0.15,
    # "callbacks": [early_stopping],
    "verbose": 0
}
tic = time()
for i in range(runs):
    iteration = {'cumulative epochs': epochs * (i + 1)}
    iter_hist = model.fit(
        X_train,
        y_pool,
        **fit_kwargs
    )

    train_loss = iter_hist.history['loss']
    weights = model.get_weights()

    if not all(np.all(np.isfinite(w)) for w in weights):
        print(f"Model became unstable at iteration {i + 1}")
        break

    y_prob = model.predict(X_val, verbose=0)

    if not np.all(np.isfinite(y_prob)):
        print(f"NaN/Inf predictions at iteration {i + 1}")
        break
        
    y_pred = np.argmax(y_prob, axis=1)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    auc = roc_auc_score(y_test, y_prob, average='macro', multi_class='ovr')
    toc = time()
    
    iteration['Training loss'] = train_loss[-1]
    iteration['Accuracy'] = acc
    iteration['macro F1'] = f1
    iteration['ROC-AUC'] = auc
    iteration['time'] = toc - tic
    history.append(iteration)
    print(f"Iteration {i+1}: ")
    for l in range(len(train_loss)):
        print(f"\t{train_loss[l]}")

run_df = pd.DataFrame(history)
run_df.round(4).to_csv(f"results/student_dropout/passive_learning.csv", index=False)

Iteration 1: 
	0.8640780448913574
	0.6533988118171692
Iteration 2: 
	0.6031410098075867
	0.5774428844451904
Iteration 3: 
	0.5616981983184814
	0.5496101975440979
Iteration 4: 
	0.5397596955299377
	0.530868411064148
Iteration 5: 
	0.5254284739494324
	0.5181592106819153
Iteration 6: 
	0.5135675072669983
	0.5080148577690125
Iteration 7: 
	0.5040453672409058
	0.5001048445701599
Iteration 8: 
	0.49652522802352905
	0.49173304438591003
Iteration 9: 
	0.4882151186466217
	0.4844587445259094
Iteration 10: 
	0.48086151480674744
	0.47841352224349976
Iteration 11: 
	0.4746659994125366
	0.4707673192024231
Iteration 12: 
	0.468472957611084
	0.4655253291130066
Iteration 13: 
	0.46251022815704346
	0.4583451747894287
Iteration 14: 
	0.45645052194595337
	0.4541664719581604
Iteration 15: 
	0.4499218165874481
	0.4474157691001892
Iteration 16: 
	0.4442249536514282
	0.4413573145866394
Iteration 17: 
	0.4383985102176666
	0.4365990459918976
Iteration 18: 
	0.43302562832832336
	0.43144509196281433
Iteration 19:

In [15]:
activation = 'leaky_relu'
params = retrieve_params(f'params/student_dropout/base_{activation}.json')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)

history = active_learning_mc(X_train, y_pool, X_val, y_test, random_sampling, num_iter=100, unlab_size=0.9, random_state=100, verbose=True,
                               hidden_units=params['hidden_layer'], activation=activation, alpha=params['leaky_relu_alpha'], out_units=n_classes, balanced=False,
                               learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'] 
)

run_df = pd.DataFrame(history)
run_df.round(4).to_csv(f"results/student_dropout/uncertainty_sampling.csv", index=False)

Iteration 1: 353 labelled instances
	Accuracy = 0.5615819209039548
	macro F1 = 0.47567788026737307
	AUC = 0.6860613262337812

Iteration 2: 385 labelled instances
	Accuracy = 0.6666666666666666
	macro F1 = 0.5035590446395342
	AUC = 0.7538927297994474

Iteration 3: 417 labelled instances
	Accuracy = 0.6779661016949152
	macro F1 = 0.5396888853932326
	AUC = 0.7800173312781339

Iteration 4: 449 labelled instances
	Accuracy = 0.6915254237288135
	macro F1 = 0.5679775703748305
	AUC = 0.7925794189991895

Iteration 5: 481 labelled instances
	Accuracy = 0.6983050847457627
	macro F1 = 0.5801014698069178
	AUC = 0.8023541218354463

Iteration 6: 513 labelled instances
	Accuracy = 0.7028248587570621
	macro F1 = 0.5964596758647653
	AUC = 0.8102467877280217

Iteration 7: 545 labelled instances
	Accuracy = 0.7096045197740113
	macro F1 = 0.6050928113872102
	AUC = 0.8167747461114435

Iteration 8: 577 labelled instances
	Accuracy = 0.7220338983050848
	macro F1 = 0.6215118965118965
	AUC = 0.8225949724903568


In [16]:
activation = 'leaky_relu'
params = retrieve_params(f'params/student_dropout/base_{activation}.json')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_pool)
X_val = scaler.transform(X_test)
for beta in [0.9]:
    history = sasla_mc(X_train, y_pool, X_val, y_test, num_iter=100, unlab_size=0.9, random_state=100, verbose=True,
                                   hidden_units=params['hidden_layer'], activation=activation, alpha=params['leaky_relu_alpha'], out_units=n_classes, balanced=False,
                                   learning_rate=params['learning_rate'], momentum=params['momentum'], weight_decay=params['weight_decay'], beta=beta)
    
    run_df = pd.DataFrame(history)
    run_df.round(4).to_csv(f"results/student_dropout/sasla_{beta}.csv", index=False)

Iteration 1: 3539 labelled instances
	Accuracy = 0.7423728813559322
	macro F1 = 0.6384317957638651
	AUC = 0.8468673691390545

Iteration 2: 3431 labelled instances
	Accuracy = 0.7435028248587571
	macro F1 = 0.6354365921543771
	AUC = 0.8653379556150451

Iteration 3: 3325 labelled instances
	Accuracy = 0.7559322033898305
	macro F1 = 0.666236186796276
	AUC = 0.8706986684627291

Iteration 4: 3324 labelled instances
	Accuracy = 0.7581920903954802
	macro F1 = 0.6726492995425025
	AUC = 0.874453766585578

Iteration 5: 3293 labelled instances
	Accuracy = 0.752542372881356
	macro F1 = 0.6689318686910865
	AUC = 0.8764532420063725

Iteration 6: 3297 labelled instances
	Accuracy = 0.7559322033898305
	macro F1 = 0.6779258570086543
	AUC = 0.876835009052067

Iteration 7: 3280 labelled instances
	Accuracy = 0.7615819209039548
	macro F1 = 0.6785944997833556
	AUC = 0.8790684539111447

Iteration 8: 3260 labelled instances
	Accuracy = 0.7581920903954802
	macro F1 = 0.6847827258390232
	AUC = 0.87993643092097